# PHA 数据科学 Agent 示例

本 Notebook 展示如何使用 PHA 的**数据科学 Agent**。

- 分析 Fitbit 等来源的健康数据。
- 根据自然语言提问制定分析方法。
- 编写并执行 Python 代码来回答问题。
- 汇总分析结果，方便用户阅读。

## 环境搭建

先安装所需依赖并配置环境。

In [ ]:
# 安装依赖关系( 需要时不做评论)
# !pip install pandas numpy jinja2 google-genai openai anthropic

In [ ]:
import os
import sys

# --- Configuration ---
# 在此设定您的 API 密钥和提供者
API_KEY = "your-api-key-here"
# 提供方选项："基因","openai","人类"
PROVIDER = "gemini"


## 加载配置和数据

通过 `Settings` 类配置数据路径和 API 密钥。

In [ ]:
from config import Settings
from pha.utils import load_persona

# 创建设置( API 键使用环境变量)
settings = Settings(
    data_dir='../data/sample',
)

print(f"Data directory: {settings.data_dir}")

In [ ]:
# 装入样本数据
summary_df, activities_df, profile_df, population_df = load_persona(settings=settings)

print(f"Summary data: {len(summary_df)} days")
print(f"Activities: {len(activities_df)} records")
print(f"\nSummary columns: {list(summary_df.columns)}")

In [ ]:
# 预览数据
print("=== Summary DataFrame (daily metrics) ===")
summary_df.head()

In [ ]:
print("=== Profile DataFrame ===")
profile_df

## 初始化数据科学 Agent

使用前面的设置创建 Agent。

In [ ]:
from pha.agents import DataScienceAgent

# 创建代理
# 隐含使用全局配置用于 LLM 后端
agent = DataScienceAgent(settings=settings)
agent.configure(api_key=API_KEY, provider=PROVIDER)  # 明确的配置

print("Data Science Agent initialized!")

## 查询健康数据

现在可以用自然语言提出关于健康数据的问题。

In [ ]:
# 示例 1:简单的统计查询
question = "What is my average daily step count over the last 30 days?"

print(f"Question: {question}")
print("\nProcessing...")

response = agent.query(question)
print(f"\nResponse:\n{response}")

In [ ]:
# 实例2:睡眠分析
question = "How does my deep sleep compare to light sleep? Show me the breakdown."

print(f"Question: {question}")
print("\nProcessing...")

response = agent.query(question)
print(f"\nResponse:\n{response}")

In [ ]:
# 实例3:关联性分析
question = "Is there a relationship between my sleep duration and my resting heart rate?"

print(f"Question: {question}")
print("\nProcessing...")

response = agent.query(question)
print(f"\nResponse:\n{response}")

## 查看分析步骤

调试或阅读实现时，可以调用 `query_with_details()` 查看所有中间步骤。

In [ ]:
question = "What's my most active day of the week based on steps?"

print(f"Question: {question}")
print("\nProcessing with details...")

result = agent.query_with_details(question)

In [ ]:
# 查看生成的方法
print("=== APPROACH ===")
print(result['approach'])

In [ ]:
# 查看生成的代码
print("=== GENERATED CODE ===")
print(result['code'])

In [ ]:
# 查看执行结果
print("=== EXECUTION RESULT ===")
print(f"Status: {result['execution_result']['status']}")
print(f"Output:\n{result['execution_result']['output']}")
if result['execution_result']['stderr']:
    print(f"Errors:\n{result['execution_result']['stderr']}")

In [ ]:
# 查看最后响应
print("=== FINAL RESPONSE ===")
print(result['response'])

## 逐步执行

也可以单独执行各个步骤，便于观察每一步的输入和结果。

In [ ]:
# 步骤1:生成方法
question = "Am I getting enough sleep compared to the general population?"
approach = agent.generate_approach(question)

print("=== APPROACH ===")
print(approach)

In [ ]:
# 步骤 2:生成代码
code = agent.generate_code(question, approach)

print("=== CODE ===")
print(code)

In [ ]:
# 步骤 3:执行代码
result = agent.execute_code(code)

print(f"Status: {result.execution_result.status}")
print(f"\nOutput:\n{result.execution_result.output}")

In [ ]:
# 步骤4:解释结果
interpretation = agent.interpret_results(
    question=question,
    approach=approach,
    code=code,
    execution_results=result.execution_result.output,
)

print("=== INTERPRETATION ===")
print(interpretation)

## 切换大模型后端

PHA 通过统一配置支持多种大模型后端。下面演示切换方式。

In [ ]:
from pha.llm import configure_global

# 切换到 OpenAI
# configure_global(provider='openai', api_key='your-openai-key')
# 代理. confition ()# 重新配置代理以使用新的后端

# 切换到 Anthropic 克劳德
# configure_global(provider='anthropic', api_key='your-anthropic-key')
# agent.configure()

## 自己试着提问

可以替换成你想用健康数据回答的问题。

In [ ]:
# 试试你自己的问题吧！
your_question = "What trends do you see in my heart rate variability?"

response = agent.query(your_question)
print(response)